In [1]:
import numpy as np
from numba import jit
from scipy.signal import find_peaks
import os
import multiprocessing as mp
import time



In [2]:
def reverser(v):
    indices = np.arange(len(v))
    rev = dict(zip(v, indices))
    return rev

In [3]:
def events_counter(w,th=0.0012,last_deletor = True):
    flag = False
    if w.size == 0:
        burst = {}
        burst['0'] = []
        burst['0'].append(0)
        return burst
    v = reverser(w)
    burst = {}
    n_burst = 0
    burst['0'] = []
    burst['0'].append(w[0])
    for i in range(1,len(w)):
        der = abs(v[(w[i])]-v[(w[i-1])])/abs(w[i]-w[i-1])
        if der < th:
            if len(burst[str(n_burst)]) < 2:
                 burst.pop(str(n_burst))
            else:
                n_burst += 1
            burst[str(n_burst)] = []
        burst[str(n_burst)].append(w[i])
    if len(burst) > 1 and last_deletor is True:
        burst.pop(str(len(burst)-1))
    list_n = []
    for n in burst:
        if len(burst[n])==1:
            flag = True
            list_n.append(n)
    for n in list_n:
        burst.pop(n)
    if flag is True:
        Burst = {}
        i = 0
        for n in burst:
            Burst[str(i)] = burst[n]
            i += 1
        return Burst
    return burst

In [4]:
def infinity_condition(last_burst,N):
    M = len(last_burst)
    if 2* last_burst[M-1]-last_burst[M-2] >= N:
        return 1
    else:
        return 0

In [5]:
jit_infinity_condition = jit(infinity_condition)

In [6]:
def baseline_before(burst,z):
    position = max(int(2*burst[0]-burst[1]),0)
    return z[position]

In [7]:
jit_baseline_before = jit(baseline_before)

In [8]:
def baseline_after(burst,z):
    M = len(burst)
    N = len(z)
    position = min(int(2*burst[M-1]-burst[M-2]),N-1)
    return z[position]

In [9]:
jit_baseline_after = jit(baseline_after)

In [10]:
def baseline_jump_onset(burst,z,baseline_value):
    min_value = np.min(z[burst[0]:burst[1]])
    return abs(baseline_value) - abs(min_value)

In [11]:
jit_baseline_jump_onset = jit(baseline_jump_onset)

In [12]:
def baseline_jump_offset(burst,z,baseline_value):
    M = len(burst)
    min_value = np.min(z[burst[M-2]:burst[M-1]])
    return abs(min_value) - abs(baseline_value)

In [13]:
jit_baseline_jump_offset = jit(baseline_jump_offset)

In [14]:
def ampl_ratio(burst,peaks,prom,mode):
    M = len(burst)
    start = np.where(peaks == burst[0])[0][0]
    end = np.where(peaks == burst[M-1])[0][0]
    if mode == 'on':
        return abs(prom[start])/max(abs(prom[start:end]))
    elif mode == 'off':
        return abs(prom[end])/max(abs(prom[start:end]))

In [15]:
def ampl_ave(burst,peaks,prom):
    M = len(burst)
    start = np.where(peaks == burst[0])[0][0]
    end = np.where(peaks == burst[M-1])[0][0]
    ave = np.average(prom[start:end])
    var = np.var(prom[start:end])
    return ave, var

In [16]:
def min_ampl_ave(burst,z):
    M = len(burst)
    min_amp_list = []
    for i in range(0,M-1):
        #start = np.where(peaks == burst[i])[0][0]
        #end = np.where(peaks == burst[i+1])[0][0]
        min_amp_list.append(min(z[burst[i]:burst[i+1]]))
        
    min_amp_list = np.array(min_amp_list)
    ave = np.average(min_amp_list)
    var = np.var(min_amp_list)

    return ave, var
    

In [17]:
def period_ratio(burst):
    M = len(burst)
    if M >= 6:
        starting = (burst[1]-burst[0])/ (burst[2]-burst[1])
        ending = (burst[M-2]-burst[M-1])/ (burst[M-3]-burst[M-2])
    elif M >= 3:
        starting = (burst[1]-burst[0])/ (burst[2]-burst[1])
        ending = 0
    else:
        starting = 0
        ending = 0
    return starting, ending

In [18]:
def amp_trend(burst,peaks,prom):
    M = len(burst)
    if M >= 3:
        peak_0 = prom[np.where(peaks == burst[0])[0][0]]
        peak_1 = prom[np.where(peaks == burst[1])[0][0]]
        peak_M_1 = prom[np.where(peaks == burst[M-1])[0][0]]
        peak_M_2 = prom[np.where(peaks == burst[M-2])[0][0]]

        starting = peak_1/ peak_0
        ending = peak_M_2/peak_M_1

    else:
        starting = 0
        ending = 0

    return starting, ending

In [19]:
def features_collector(z,peak,burst,prom):
    features_vector = np.zeros(36)
    M = len(burst)
    features_vector[0] = M
    if len(burst) == 1:
        features_vector[1] = infinity_condition(burst['0'],len(z))
        features_vector[2] = len(burst['0'])
        features_vector[3] = jit_baseline_before(burst['0'],z)
        features_vector[4] = jit_baseline_after(burst['0'],z)
        features_vector[5] = jit_baseline_jump_onset(burst['0'],z,features_vector[3])
        features_vector[6] = jit_baseline_jump_offset(burst['0'],z,features_vector[4])
        features_vector[7] = ampl_ratio(burst['0'],peak,prom,'on')
        features_vector[8] = ampl_ratio(burst['0'],peak,prom,'off')
        features_vector[9],features_vector[10] = ampl_ave(burst['0'],peak,prom)
        features_vector[11], features_vector[12] = burst['0'][0], burst['0'][-1]
        features_vector[13],features_vector[14] = min_ampl_ave(burst['0'],z)
        features_vector[15],features_vector[16] = period_ratio(burst['0'])
        features_vector[17],features_vector[18] = amp_trend(burst['0'],peak,prom)
    elif len(burst) >= 4 and len(burst[str(M-2)])/len(burst[str(M-1)]) >= 1:
        features_vector[1] = infinity_condition(burst[str(M-1)],len(z))
        features_vector[2] = len(burst[str(M-2)])
        features_vector[3] = jit_baseline_before(burst[str(M-2)],z)
        features_vector[4] = jit_baseline_after(burst[str(M-2)],z)
        features_vector[5] = jit_baseline_jump_onset(burst[str(M-2)],z,features_vector[3])
        features_vector[6] = jit_baseline_jump_offset(burst[str(M-2)],z,features_vector[4])
        features_vector[7] = ampl_ratio(burst[str(M-2)],peak,prom,'on')
        features_vector[8] = ampl_ratio(burst[str(M-2)],peak,prom,'off')
        features_vector[9],features_vector[10] = ampl_ave(burst[str(M-2)],peak,prom)
        features_vector[11],features_vector[12] = burst[str(M-2)][0], burst[str(M-2)][-1]
        features_vector[13],features_vector[14] = min_ampl_ave(burst[str(M-2)],z)
        features_vector[15],features_vector[16] = period_ratio(burst[str(M-2)])
        features_vector[17],features_vector[18] = amp_trend(burst[str(M-2)],peak,prom)
        features_vector[19] = len(burst[str(M-3)])
        features_vector[20] = jit_baseline_before(burst[str(M-3)],z)
        features_vector[21] = jit_baseline_after(burst[str(M-3)],z)
        features_vector[22] = jit_baseline_jump_onset(burst[str(M-3)],z,features_vector[3])
        features_vector[23] = jit_baseline_jump_offset(burst[str(M-3)],z,features_vector[4])
        features_vector[24] = ampl_ratio(burst[str(M-3)],peak,prom,'on')
        features_vector[25] = ampl_ratio(burst[str(M-3)],peak,prom,'off')
        features_vector[26],features_vector[27] = ampl_ave(burst[str(M-3)],peak,prom)
        features_vector[28], features_vector[29] = burst[str(M-2)][0], burst[str(M-3)][-1]
        features_vector[30],features_vector[31] = min_ampl_ave(burst[str(M-3)],z)
        features_vector[32],features_vector[33] = period_ratio(burst[str(M-3)])
        features_vector[34],features_vector[35] = amp_trend(burst[str(M-3)],peak,prom)
    else:
        features_vector[1] = infinity_condition(burst[str(M-1)],len(z))
        features_vector[2] = len(burst[str(M-1)])
        features_vector[3] = jit_baseline_before(burst[str(M-1)],z)
        features_vector[4] = jit_baseline_after(burst[str(M-1)],z)
        features_vector[5] = jit_baseline_jump_onset(burst[str(M-1)],z,features_vector[3])
        features_vector[6] = jit_baseline_jump_offset(burst[str(M-1)],z,features_vector[4])
        features_vector[7] = ampl_ratio(burst[str(M-1)],peak,prom,'on')
        features_vector[8] = ampl_ratio(burst[str(M-1)],peak,prom,'off')
        features_vector[9],features_vector[10] = ampl_ave(burst[str(M-1)],peak,prom)
        features_vector[11],features_vector[12] = burst[str(M-1)][0], burst[str(M-1)][-1]
        features_vector[13],features_vector[14] = min_ampl_ave(burst[str(M-1)],z)
        features_vector[15],features_vector[16] = period_ratio(burst[str(M-1)])
        features_vector[17],features_vector[18] = amp_trend(burst[str(M-1)],peak,prom)
        features_vector[19] = len(burst[str(M-2)])
        features_vector[20] = jit_baseline_before(burst[str(M-2)],z)
        features_vector[21] = jit_baseline_after(burst[str(M-2)],z)
        features_vector[22] = jit_baseline_jump_onset(burst[str(M-2)],z,features_vector[3])
        features_vector[23] = jit_baseline_jump_offset(burst[str(M-2)],z,features_vector[4])
        features_vector[24] = ampl_ratio(burst[str(M-2)],peak,prom,'on')
        features_vector[25] = ampl_ratio(burst[str(M-2)],peak,prom,'off')
        features_vector[26],features_vector[27] = ampl_ave(burst[str(M-2)],peak,prom)
        features_vector[28], features_vector[29] = burst[str(M-2)][0], burst[str(M-2)][-1]
        features_vector[30],features_vector[31] = min_ampl_ave(burst[str(M-2)],z)
        features_vector[32],features_vector[33] = period_ratio(burst[str(M-1)])
        features_vector[34],features_vector[35] = amp_trend(burst[str(M-2)],peak,prom)

    return features_vector

In [20]:
def Kbath_ode_model(params):
    t_init = 0
    t_final = 10000
    dt = 0.01
    gamma = 0.04
    epsilon = 0.01 
    ts = np.arange(t_init, t_final, dt)
    v_0 = -78.0
    n_inf = 1.0 / (1.0 + np.exp((-19.0 - v_0) / 18.0))
    z0 = np.array([-78.0, n_inf, -0.6, 0.8])
    Np = 10000
    t = np.linspace(0, Np, int(Np / 0.01))
    v = np.zeros_like(t)
    n = np.zeros_like(t)
    DK_i = np.zeros_like(t)
    Kg = np.zeros_like(t)
    Cm = 1.0
    tau_n = 0.25

    beta = 3.
    w_i = 2160.
    w_o = w_i / beta
    rho = 250.

    K_bath = params[0]

    Na_i0 = 16.0
    Na_o0 = 138.0
    K_i0 = 140.0
    K_o0 = 4.80
    Cl_o0 = 112.0
    Cl_i0 = 5.0
    g_Cl = params[1]
    g_Na = params[2]
    g_K = params[3]
    g_Nal = 0.02
    g_Kl = 0.12
    v[0] = z0[0]
    n[0] = z0[1]
    DK_i[0] = z0[2]
    Kg[0] = z0[3]

    for i in range(1, ts.shape[0]):
        # k1
        DNa_i = -DK_i[i-1]
        DNa_o = -beta * DNa_i
        DK_o = -beta * DK_i[i-1]
        K_i = K_i0 + DK_i[i-1]
        Na_i = Na_i0 + DNa_i
        Na_o = Na_o0 + DNa_o
        K_o = K_o0 + DK_o + Kg[i-1]

        m_inf = 1.0 / (1.0 + np.exp((-24.0 - v[i-1]) / 12.0))
        n_inf = 1.0 / (1.0 + np.exp((-19.0 - v[i-1]) / 18.0))
        h_n = 1.1 - 1.0 / (1.0 + np.exp(-8.0 * (n[i-1] - 0.4)))

        I_Na = (g_Nal + g_Na * m_inf * h_n) * (v[i-1] - 26.64 * np.log(Na_o / Na_i))
        I_K = (g_Kl + g_K * n[i-1]) * (v[i-1] - 26.64 * np.log(K_o / K_i))
        I_Cl = g_Cl * (v[i-1] + 26.64 * np.log(Cl_o0 / Cl_i0))
        I_pump = rho * (1.0 / (1.0 + np.exp((21.0 - Na_i) / 2.0))) * (1.0 / (1.0 + np.exp((5.5 - K_o))))

        k1_v = (-1.0 / Cm) * (I_Cl + I_Na + I_K + I_pump)
        k1_n = (n_inf - n[i-1]) / tau_n
        k1_DK_i = -(gamma / w_i) * (I_K - 2.0 * I_pump)
        k1_Kg = epsilon * (K_bath - K_o)

        # k2
        v_mid = v[i-1] + dt/2 * k1_v
        n_mid = n[i-1] + dt/2 * k1_n
        DK_i_mid = DK_i[i-1] + dt/2 * k1_DK_i
        Kg_mid = Kg[i-1] + dt/2 * k1_Kg

        DNa_i = -DK_i_mid
        DNa_o = -beta * DNa_i
        DK_o = -beta * DK_i_mid
        K_i = K_i0 + DK_i_mid
        Na_i = Na_i0 + DNa_i
        Na_o = Na_o0 + DNa_o
        K_o = K_o0 + DK_o + Kg_mid

        m_inf = 1.0 / (1.0 + np.exp((-24.0 - v_mid) / 12.0))
        n_inf = 1.0 / (1.0 + np.exp((-19.0 - v_mid) / 18.0))
        h_n = 1.1 - 1.0 / (1.0 + np.exp(-8.0 * (n_mid - 0.4)))

        I_Na = (g_Nal + g_Na * m_inf * h_n) * (v_mid - 26.64 * np.log(Na_o / Na_i))
        I_K = (g_Kl + g_K * n_mid) * (v_mid - 26.64 * np.log(K_o / K_i))
        I_Cl = g_Cl * (v_mid + 26.64 * np.log(Cl_o0 / Cl_i0))
        I_pump = rho * (1.0 / (1.0 + np.exp((21.0 - Na_i) / 2.0))) * (1.0 / (1.0 + np.exp((5.5 - K_o))))

        k2_v = (-1.0 / Cm) * (I_Cl + I_Na + I_K + I_pump)
        k2_n = (n_inf - n_mid) / tau_n
        k2_DK_i = -(gamma / w_i) * (I_K - 2.0 * I_pump)
        k2_Kg = epsilon * (K_bath - K_o)

        # k3
        v_mid = v[i-1] + dt/2 * k2_v
        n_mid = n[i-1] + dt/2 * k2_n
        DK_i_mid = DK_i[i-1] + dt/2 * k2_DK_i
        Kg_mid = Kg[i-1] + dt/2 * k2_Kg

        DNa_i = -DK_i_mid
        DNa_o = -beta * DNa_i
        DK_o = -beta * DK_i_mid
        K_i = K_i0 + DK_i_mid
        Na_i = Na_i0 + DNa_i
        Na_o = Na_o0 + DNa_o
        K_o = K_o0 + DK_o + Kg_mid

        m_inf = 1.0 / (1.0 + np.exp((-24.0 - v_mid) / 12.0))
        n_inf = 1.0 / (1.0 + np.exp((-19.0 - v_mid) / 18.0))
        h_n = 1.1 - 1.0 / (1.0 + np.exp(-8.0 * (n_mid - 0.4)))

        I_Na = (g_Nal + g_Na * m_inf * h_n) * (v_mid - 26.64 * np.log(Na_o / Na_i))
        I_K = (g_Kl + g_K * n_mid) * (v_mid - 26.64 * np.log(K_o / K_i))
        I_Cl = g_Cl * (v_mid + 26.64 * np.log(Cl_o0 / Cl_i0))
        I_pump = rho * (1.0 / (1.0 + np.exp((21.0 - Na_i) / 2.0))) * (1.0 / (1.0 + np.exp((5.5 - K_o))))

        k3_v = (-1.0 / Cm) * (I_Cl + I_Na + I_K + I_pump)
        k3_n = (n_inf - n_mid) / tau_n
        k3_DK_i = -(gamma / w_i) * (I_K - 2.0 * I_pump)
        k3_Kg = epsilon * (K_bath - K_o)

        # k4
        v_end = v[i-1] + dt * k3_v
        n_end = n[i-1] + dt * k3_n
        DK_i_end = DK_i[i-1] + dt * k3_DK_i
        Kg_end = Kg[i-1] + dt * k3_Kg

        DNa_i = -DK_i_end
        DNa_o = -beta * DNa_i
        DK_o = -beta * DK_i_end
        K_i = K_i0 + DK_i_end
        Na_i = Na_i0 + DNa_i
        Na_o = Na_o0 + DNa_o
        K_o = K_o0 + DK_o + Kg_end

        m_inf = 1.0 / (1.0 + np.exp((-24.0 - v_end) / 12.0))
        n_inf = 1.0 / (1.0 + np.exp((-19.0 - v_end) / 18.0))
        h_n = 1.1 - 1.0 / (1.0 + np.exp(-8.0 * (n_end - 0.4)))

        I_Na = (g_Nal + g_Na * m_inf * h_n) * (v_end - 26.64 * np.log(Na_o / Na_i))
        I_K = (g_Kl + g_K * n_end) * (v_end - 26.64 * np.log(K_o / K_i))
        I_Cl = g_Cl * (v_end + 26.64 * np.log(Cl_o0 / Cl_i0))
        I_pump = rho * (1.0 / (1.0 + np.exp((21.0 - Na_i) / 2.0))) * (1.0 / (1.0 + np.exp((5.5 - K_o))))

        k4_v = (-1.0 / Cm) * (I_Cl + I_Na + I_K + I_pump)
        k4_n = (n_inf - n_end) / tau_n
        k4_DK_i = -(gamma / w_i) * (I_K - 2.0 * I_pump)
        k4_Kg = epsilon * (K_bath - K_o)

        # Aggiornamento delle variabili con RK4
        v[i] = v[i-1] + (dt/6) * (k1_v + 2*k2_v + 2*k3_v + k4_v)
        n[i] = n[i-1] + (dt/6) * (k1_n + 2*k2_n + 2*k3_n + k4_n)
        DK_i[i] = DK_i[i-1] + (dt/6) * (k1_DK_i + 2*k2_DK_i + 2*k3_DK_i + k4_DK_i)
        Kg[i] = Kg[i-1] + (dt/6) * (k1_Kg + 2*k2_Kg + 2*k3_Kg + k4_Kg)

    return v

In [21]:
Kbath_ode_model_jit = jit(Kbath_ode_model)

In [22]:
def simulation_features_collector(kb,g_Cl,g_Na,g_K):

    t_init = 0
    t_final = 10000
    dt = 0.01
    ts = np.arange(t_init, t_final, dt)
    nt = ts.shape[0]
    params = [kb, g_Cl, g_Na, g_K]

    z = Kbath_ode_model_jit(params)

    peaks,b = find_peaks(z[0:nt],prominence=0.9)
    prom = b['prominences']
    burst = events_counter(peaks,last_deletor=False)
    if len(peaks) <= 4 or not burst:
        vec = np.zeros(36)
        vec[0] = len(peaks)
        return vec
    else:
        return features_collector(z,peaks,burst,prom)


In [23]:
def writer(path,kb,g_Na_space,g_Cl_space,g_K_space):
    path = path + str(kb) + '.txt'
    with open(path, 'w') as t:
            flag = True
            for i in range(0,len(g_Na_space)):
                for j in range(0,len(g_Cl_space)):
                     for k in range(0,len(g_K_space)):
                        V = simulation_features_collector(kb =kb ,g_Cl = g_Cl_space[j],g_Na = g_Na_space[i],g_K = g_K_space[k])
                        if flag is True:
                            t.write('kb' +  '\t' + 'g_Na' + '\t' + 'g_Cl' + '\t' + 'g_K' + '\t' + 
                                                    'Num_burst' + '\t' + 'INF_COND' + '\t' + 'Num_Spike_1' + '\t' + 'Baseline_b_1' + '\t' + 
                                                    'Baseline_a_1' + '\t' + 'Baseline_jump_on_1' + '\t' + 'Baseline_jump_off_1' + '\t' + 
                                                    'amplitude_ratio_on_1' + '\t' + 'amplitude_ratio_off_1' + '\t' + 'ave_amp_1' + '\t' + 
                                                    'var_amp_1' + '\t'+ 'start_1'+'\t'+'end_1'+'\t'+'ave_min_1'+'\t'+'var_min_1'+'\t'+ 'starting_T_1' +'\t'+ 'ending_T_1' +'\t'+ 'starting_A_1' +'\t'+ 'ending_A_1'+'\t'+'Num_Spike_2' + '\t' + 'Baseline_b_2' + '\t' + 
                                                    'Baseline_a_2' + '\t' + 'Baseline_jump_on_2' + '\t' + 'Baseline_jump_off_2' + '\t' + 
                                                    'amplitude_ratio_on_2' + '\t' + 'amplitude_ratio_off_2' + '\t' + 'ave_amp_2' + '\t' + 
                                                    'var_amp_2'+ '\t'+ 'start_2'+'\t'+'end_2'+'\t'+'ave_min_2'+'\t'+'var_min_2'+ '\t'+ 'starting_T_2' +'\t'+ 'ending_T_2' +'\t'+ 'starting_A_2' +'\t'+ 'ending_A_2'+ '\n')
                            flag = False
                        t.write(str(kb) +  '\t' + str(g_Na_space[i]) + '\t' + str(g_Cl_space[j]) + '\t' + str(g_K_space[k]) + '\t' + 
                                                str(V[0]) + '\t' + str(V[1]) + '\t' + str(V[2]) + '\t' + str(V[3]) + '\t' + 
                                                str(V[4]) + '\t' + str(V[5]) + '\t' + str(V[6]) + '\t' + 
                                                str(V[7]) + '\t' + str(V[8]) + '\t' + str(V[9]) + '\t' + 
                                                str(V[10]) + '\t' + str(V[11]) + '\t' + str(V[12]) + '\t' + 
                                                str(V[13]) + '\t' + str(V[14]) + '\t' + str(V[15]) + '\t' + 
                                                str(V[16]) + '\t' + str(V[17]) + '\t' + str(V[18]) + '\t' + 
                                                str(V[19]) + '\t' + str(V[20]) + '\t' + str(V[21]) + '\t' + 
                                                str(V[22]) + '\t' + str(V[23]) + '\t' + str(V[24]) + '\t' + 
                                                str(V[25]) + '\t' + str(V[26]) + '\t' + str(V[27]) + '\t' + 
                                                str(V[28]) + '\t' + str(V[29]) + '\t' + str(V[30]) + '\t' + str(V[31]) + '\t' + str(V[32]) + '\t' + str(V[33]) + '\t' + str(V[34]) + '\t' + str(V[35]) + '\n')

In [24]:
g_Cl = 7.5
g_Na = 40.0
g_K = 22.0
g_Na_space = np.linspace(g_Na-g_Na/3,g_Na+g_Na/3,10)
g_Cl_space = np.linspace(g_Cl-g_Cl/3,g_Cl+g_Cl/3,10)
g_K_space = np.linspace(g_K-g_K/3,g_K+g_K/3,5)

In [25]:
path = "/Users/gianmarcocafaro/Desktop/simulazioni_finali/4ritest_kb_"
KB =np.linspace(7.5, 17.5, 20)
for kb in KB:
    writer(path,kb,g_Na_space,g_Cl_space,g_K_space)
    print(kb)

7.5
8.026315789473685
8.552631578947368
9.078947368421053
9.605263157894736
10.131578947368421
10.657894736842106
11.18421052631579
11.710526315789473
12.236842105263158
12.763157894736842
13.289473684210526
13.81578947368421
14.342105263157894
14.868421052631579
15.394736842105264
15.921052631578947
16.44736842105263
16.973684210526315
17.5
